# Русский wake-word «Помогатор»

Чистый повторяемый pipeline: русские Piper-голоса → positive/hard-negative WAV → шум/эхо → openWakeWord ONNX. Результат предназначен для личного некоммерческого POC. Запускайте ячейки строго сверху вниз.

In [ ]:
# 1. Репозиторий, закреплённые зависимости и проверка окружения
from pathlib import Path
import subprocess
if not Path('ph-server').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feature/vosk-local-wakeword', 'https://github.com/IavnFGV/ph-server.git'], check=True)
else:
    subprocess.run(['git', '-C', 'ph-server', 'pull', '--ff-only'], check=True)
subprocess.run(['python', 'ph-server/training/pomogator/colab_pipeline.py', 'setup'], check=True)

In [ ]:
# 2. Русские positive и hard-negative примеры
subprocess.run(['python', 'ph-server/training/pomogator/colab_pipeline.py', 'generate'], check=True)
from IPython.display import Audio, display
example = next(Path('my_custom_model/pomogator/positive_test').glob('*.wav'))
print('Прослушайте контрольный пример:', example)
display(Audio(str(example)))

In [ ]:
# 3. Закреплённые RIR, AudioSet, FMA и negative features (~18 ГБ)
subprocess.run(['python', 'ph-server/training/pomogator/colab_pipeline.py', 'data'], check=True)

In [ ]:
# 4. Compatibility patches, аугментация, обучение, проверка и скачивание
subprocess.run(['python', 'ph-server/training/pomogator/colab_pipeline.py', 'train'], check=True)
from google.colab import files
for artifact in ('my_custom_model/pomogator.onnx', 'my_custom_model/pomogator-checkpoint.pt', 'my_custom_model/training-report.txt', 'pomogator-augment_clips.log', 'pomogator-train_model.log'):
    files.download(artifact)